# Finetuning

This notebook is self-contained and should be run on https://jupyter-hub.io-ancotel.local:8443.

In [ ]:
import json
import os
import subprocess
from collections import Counter
from functools import partial
from pathlib import Path
from typing import Any

In [ ]:
# GPU configuration
GPU_MODE = "single"  # either "single" (best free GPU), "multi" (N best free GPUs), "all" (every GPU)
NUM_GPUS = 2  # only used when GPU_MODE="multi"

_smi = subprocess.run(
    ["/usr/bin/nvidia-smi", "--query-gpu=memory.free", "--format=csv,nounits,noheader"],
    capture_output=True,
    text=True,
    check=True,
)
_free = [int(x) for x in _smi.stdout.strip().splitlines()]

if GPU_MODE == "single":
    _gpu = _free.index(max(_free))
    os.environ["CUDA_VISIBLE_DEVICES"] = str(_gpu)
    print(f"Using physical GPU {_gpu} ({_free[_gpu]} MiB free)")
elif GPU_MODE == "multi":
    _ranked = sorted(range(len(_free)), key=lambda i: _free[i], reverse=True)
    _selected = _ranked[:NUM_GPUS]
    os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in _selected)
    print(
        f"Using physical GPUs {_selected} (free: {[_free[g] for g in _selected]} MiB)"
    )
elif GPU_MODE == "all":
    print(f"Using all {len(_free)} GPUs (free: {_free} MiB)")
else:
    msg = f"Unknown GPU_MODE: {GPU_MODE!r}"
    raise ValueError(msg)

In [ ]:
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [ ]:
# Configuration
CONFIG = {
    "paths": {
        "data": Path("../data/processed"),
        "checkpoint_dir": "Qwen2.5-7B-Instruct-CP",
        "ft_model": "Qwen2.5-7B-Instruct-FT",
    },
    "model": {
        "base_model": "Qwen/Qwen2.5-7B-Instruct",
        "use_qlora": False,
        "attn_implementation": "flash_attention_2",
        "torch_compile": False,
    },
    "lora": {
        "rank": 16,
        "alpha": 32,
        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        "dropout": 0.05,
        "use_rslora": False,
    },
    "training": {
        "per_device_train_batch_size": 8,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-5,
        "weight_decay": 0.05,
        "num_train_epochs": 3,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.03,
        "max_length": 4096,
        "dynamic_padding": True,
        "early_stopping_patience": 5,
        "neftune_noise_alpha": 5,
    },
    "seed": 42,
}

In [ ]:
# Extract commonly used values
BASE_MODEL = CONFIG["model"]["base_model"]
SEED = CONFIG["seed"]

In [ ]:
# Set seed for reproducibility
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)

In [ ]:
# Enable TF32 for faster matmul on Ampere+ GPUs
torch.backends.cuda.matmul.fp32_precision = "tf32"
torch.backends.cudnn.conv.fp32_precision = "tf32"

## Read input data

In [ ]:
# Create dataset from the jsonl file
dataset = load_dataset(
    "json",
    data_files="train_prompt_examples_c250_r10.jsonl",
)["train"]

In [ ]:
# Train-eval split of the dataset
dataset = dataset.train_test_split(test_size=0.1, seed=SEED)

## Tokenizer

In [ ]:
def format_with_chat_template(
    batch: dict[str, list[dict[str, str]]],
    tokenizer: AutoTokenizer,
) -> dict[str, list[str]]:
    """Return the full chat prompt as a single string.

    Args:
        batch: A batch from the dataset, containing chat messages.
        tokenizer: The tokenizer to use for formatting the chat template.

    Returns:
        A dictionary with a "text" key containing the formatted chat prompts.

    """
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        for messages in batch["messages"]
    ]
    return {"text": texts}

In [ ]:
# Build tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

In [ ]:
# Inspect special tokens
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"BOS token: {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

In [ ]:
# Add padding token if needed
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

In [ ]:
# Apply formatting to the existing dataset
dataset_formatted = dataset.map(
    partial(format_with_chat_template, tokenizer=tokenizer),
    batched=True,
    desc="Formatting with chat template",
)

In [ ]:
# Extract train and eval formatted dataset
train_dataset_formatted = dataset_formatted["train"]
eval_dataset_formatted = dataset_formatted["test"]

## Tokenize the text

In [ ]:
def detect_assistant_separator(tokenizer: AutoTokenizer) -> str:
    """Return the separator string for the assistant role based on the chat template.

    Args:
        tokenizer: The tokenizer with chat template support.

    Returns:
        The assistant separator string.

    Raises:
        ValueError: If the assistant separator could not be detected.

    """
    test_messages = [
        {"role": "system", "content": "test"},
        {"role": "user", "content": "test"},
        {"role": "assistant", "content": "test"},
    ]
    formatted = tokenizer.apply_chat_template(test_messages, tokenize=False)

    # Common patterns for different models
    patterns = {
        "llama3": "<|start_header_id|>assistant<|end_header_id|>\n\n",
        "qwen": "<|im_start|>assistant\n",
        "phi3": "<|assistant|>\n",
        "gemma": "model\n",
        "mistral": "[/INST]",  # Mistral uses [INST] and [/INST]
    }

    # Find whether the pattern exists in the formatted text
    for model_name, pattern in patterns.items():
        if pattern in formatted:
            print(
                f"Detected {model_name} chat template, using separator: {pattern!r}",
            )
            return pattern

    # Fallback, look for the assistant marker
    if "<|start_header_id|>assistant<|end_header_id|>" in formatted:
        return "<|start_header_id|>assistant<|end_header_id|>\n\n"
    if "<|im_start|>assistant" in formatted:
        return "<|im_start|>assistant\n"
    if "<|assistant|>" in formatted:
        return "<|assistant|>\n"
    if "[/INST]" in formatted:
        return "[/INST]"

    # If nothing found, print formatted text for manual inspection
    msg = (
        "Could not automatically detect the assistant separator.\n"
        "Formatted test messages:\n"
        f"{formatted}\n"
        "Please inspect the above output to determine the correct separator."
    )
    raise ValueError(msg)

In [ ]:
def tokenize_with_mask_batch(
    batch: dict[str, list[str] | str],
    tokenizer: AutoTokenizer,
    max_length: int,
    sep: str = "<|im_start|>assistant\n",
) -> dict[str, list[list[int]]]:
    """Return tokenized inputs and labels with prompt tokens masked.

    Splits each text at ``sep`` so that everything up to and including the
    separator is treated as the prompt (masked with -100 in labels), and
    everything after is the assistant response the model learns to generate.
    Sequences are truncated to ``max_length`` but not padded — padding is
    handled by the data collator.

    Args:
        batch: A dict from the dataset.
               When batched=True, contains "text" key with a list of strings.
               When batched=False, contains "text" key with a single string.
        tokenizer: The tokenizer to use for tokenization.
        max_length: The maximum sequence length for truncation.
        sep: The separator string between user and assistant messages.

    Returns:
        A dictionary with tokenized "input_ids", "attention_mask", and "labels".

    """
    texts = batch["text"]

    # Handle both single string (batched=False) and list of strings (batched=True)
    if isinstance(texts, str):
        texts = [texts]

    all_input_ids: list[list[int]] = []
    all_labels: list[list[int]] = []
    all_attention_masks: list[list[int]] = []

    for text in texts:
        # Split at the separator: everything up to and including sep is
        # the prompt (masked), everything after is the training target.
        parts = text.split(sep, maxsplit=1)
        if len(parts) != 2:
            msg = f"Separator {sep!r} not found in text"
        raise ValueError(msg)
        input_text = parts[0] + sep
        assistant_text = parts[1]

        # Tokenize segments
        input_ids = tokenizer(input_text, add_special_tokens=False).input_ids
        assistant_ids = tokenizer(assistant_text, add_special_tokens=False).input_ids

        # Concatenate
        full_ids = input_ids + assistant_ids

        # Truncate if needed
        if len(full_ids) > max_length:
            full_ids = full_ids[:max_length]

        # Build labels — mask system + user tokens
        labels = full_ids.copy()
        for i in range(min(len(input_ids), len(full_ids))):
            labels[i] = -100

        attention_mask = [1] * len(full_ids)

        all_input_ids.append(full_ids)
        all_labels.append(labels)
        all_attention_masks.append(attention_mask)

    return {
        "input_ids": all_input_ids,
        "labels": all_labels,
        "attention_mask": all_attention_masks,
    }

In [ ]:
# Auto-detect based on chat template
sep = detect_assistant_separator(tokenizer)

In [ ]:
# Apply tokenization to the existing dataset
dataset_tokenized = dataset_formatted.map(
    partial(
        tokenize_with_mask_batch,
        tokenizer=tokenizer,
        max_length=CONFIG["training"]["max_length"],
        sep=sep,
    ),
    batched=True,
    batch_size=100,
    remove_columns=train_dataset_formatted.column_names,
    desc="Tokenizing",
    num_proc=4,
)

In [ ]:
# Extract train and eval tokenized datasets
train_dataset_tokenized = dataset_tokenized["train"]
eval_dataset_tokenized = dataset_tokenized["test"]

In [ ]:
# Sequence length distribution (actual content, no padding)
_lengths = [len(ex["input_ids"]) for ex in train_dataset_tokenized]
_lengths_eval = [len(ex["input_ids"]) for ex in eval_dataset_tokenized]
_all_lengths = _lengths + _lengths_eval

print(f"Sequence length statistics (train + eval, n={len(_all_lengths)}):")
print(f"  Min:  {min(_all_lengths):,}")
print(f"  Mean: {np.mean(_all_lengths):,.0f}")
print(f"  P50:  {int(np.percentile(_all_lengths, 50)):,}")
print(f"  P90:  {int(np.percentile(_all_lengths, 90)):,}")
print(f"  P95:  {int(np.percentile(_all_lengths, 95)):,}")
print(f"  P99:  {int(np.percentile(_all_lengths, 99)):,}")
print(f"  Max:  {max(_all_lengths):,}")
print(f"  max_length: {CONFIG['training']['max_length']:,}")
_truncated = sum(
    1 for length in _all_lengths if length >= CONFIG["training"]["max_length"]
)
print(
    f"  Truncated: {_truncated}/{len(_all_lengths)} ({_truncated / len(_all_lengths):.1%})"
)

In [ ]:
# Quick decoding preview
for i in range(1):
    ex = train_dataset_tokenized[i]
    input_ids = ex["input_ids"]
    labels = ex["labels"]

    # Decode entire input
    decoded_all = tokenizer.decode(input_ids, skip_special_tokens=False)

    # Decode only the trainable (non-masked) tokens
    trainable_ids = [
        tid for tid, lbl in zip(input_ids, labels, strict=True) if lbl != -100
    ]
    decoded_trainable = tokenizer.decode(trainable_ids, skip_special_tokens=False)

    # Print results
    print(f"Full decoded text (example {i}):\n{decoded_all}\n")
    print(f"Decoded trainable region (label != -100):\n{decoded_trainable}\n")
    print(
        f"Summary: Total tokens: {len(labels)}, "
        f"Trainable: {sum(lbl != -100 for lbl in labels)}, "
        f"Masked: {sum(lbl == -100 for lbl in labels)}",
    )

## PEFT

In [ ]:
# Prepare the model for training with optional 4-bit quantization
if CONFIG["model"]["use_qlora"]:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

In [ ]:
# Load pretrained base model
if CONFIG["model"]["use_qlora"]:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        dtype=torch.bfloat16,
        attn_implementation=CONFIG["model"]["attn_implementation"],
        device_map="auto",
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.bfloat16,
        attn_implementation=CONFIG["model"]["attn_implementation"],
        device_map="auto",
        trust_remote_code=True,
    )

In [ ]:
# Dont' use cache
model.config.use_cache = False  # Important for gradient checkpointing

In [ ]:
# Sync pad, bos and eos token IDs with the tokenizer
for cfg in (model.config, model.generation_config):
    cfg.pad_token_id = tokenizer.pad_token_id
    cfg.bos_token_id = tokenizer.bos_token_id
    cfg.eos_token_id = tokenizer.eos_token_id

In [ ]:
# Check the embedding sizes
vocab_len = len(tokenizer)
embed_len = model.get_input_embeddings().weight.size(0)
if vocab_len > embed_len:
    print("Mismatch between vocabulary and embeddings size. Adding extra rows.")
    model.resize_token_embeddings(vocab_len)
else:
    print(f"Tokenizer={vocab_len}, embed_rows={embed_len} (extra rows reserved).")

In [ ]:
# Optionally prepare the model for QLoRA in the case of 4-bit quantization
if CONFIG["model"]["use_qlora"]:
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    model = prepare_model_for_kbit_training(model)

In [ ]:
# Inspect model architecture to see available modules for (Q)LoRA
for name, module in model.named_modules():
    if any(proj in name for proj in ["proj", "mlp", "attn"]):
        print(f"{name}: {type(module).__name__}")

In [ ]:
# Define LoRA configuration
lora_config = LoraConfig(
    r=CONFIG["lora"]["rank"],
    lora_alpha=CONFIG["lora"]["alpha"],
    target_modules=CONFIG["lora"]["target_modules"],
    lora_dropout=CONFIG["lora"]["dropout"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    use_rslora=CONFIG["lora"]["use_rslora"],
)

In [ ]:
# Add PEFT LoRA adapter
model = get_peft_model(model, lora_config)

In [ ]:
# Compile the model for faster training (PyTorch 2.9+, set to False if issues arise)
if CONFIG["model"]["torch_compile"]:
    model = torch.compile(model, dynamic=True)
    print("Model compiled with torch.compile(dynamic=True)")

In [ ]:
# Parameter devices
param_devices = {p.device for p in model.parameters()}
print(param_devices)

In [ ]:
# Parameter distribution across devices
cnt = Counter(str(p.device) for p in model.parameters() if p.requires_grad)
dict(cnt)

In [ ]:
# Percentage of trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable {trainable:,} / Total {total:,} -> {trainable / total:.2%}")

In [ ]:
# Set training arguments
training_args = TrainingArguments(
    output_dir=CONFIG["paths"]["checkpoint_dir"],
    overwrite_output_dir=True,
    remove_unused_columns=False,
    per_device_train_batch_size=CONFIG["training"]["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["training"]["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["training"]["gradient_accumulation_steps"],
    learning_rate=CONFIG["training"]["learning_rate"],
    weight_decay=CONFIG["training"]["weight_decay"],
    num_train_epochs=CONFIG["training"]["num_train_epochs"],
    lr_scheduler_type=CONFIG["training"]["lr_scheduler_type"],
    warmup_ratio=CONFIG["training"]["warmup_ratio"],
    logging_strategy="steps",
    logging_first_step=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    neftune_noise_alpha=CONFIG["training"]["neftune_noise_alpha"],
    optim="paged_adamw_8bit" if CONFIG["model"]["use_qlora"] else "adamw_torch_fused",
    ddp_find_unused_parameters=False,
    dataloader_num_workers=4,
    seed=SEED,
)

In [ ]:
def causal_collator(features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    """Pad and collate pre-tokenized features into a batch of tensors.

    When ``dynamic_padding`` is enabled, sequences are padded to the longest
    sequence in the batch.  Otherwise they are padded to ``max_length``.

    Args:
        features: A list of feature dictionaries, each containing lists of
            integers for keys like "input_ids", "attention_mask", and "labels".

    Returns:
        A dictionary with keys mapping to batched ``torch.Tensor`` objects.

    """
    if CONFIG["training"]["dynamic_padding"]:
        target_len = max(len(f["input_ids"]) for f in features)
    else:
        target_len = CONFIG["training"]["max_length"]

    batch = {}
    for key in features[0]:
        if key == "labels":
            pad_value = -100
        elif key == "attention_mask":
            pad_value = 0
        else:
            pad_value = tokenizer.pad_token_id
        padded = [f[key] + [pad_value] * (target_len - len(f[key])) for f in features]
        batch[key] = torch.tensor(padded, dtype=torch.long)
    return batch

In [ ]:
def preprocess_logits_for_metrics(
    logits: torch.Tensor,
    labels: torch.Tensor,  # noqa: ARG001
) -> torch.Tensor:
    """Reduce logits to argmax predictions to save memory during evaluation."""
    return logits.argmax(dim=-1)


def compute_metrics(eval_pred: tuple[np.ndarray, np.ndarray]) -> dict[str, float]:
    """Compute token-level accuracy on the assistant response tokens."""
    preds, labels = eval_pred
    # Shift for causal LM: logits at position i predict token at position i+1
    preds = preds[:, :-1]
    labels = labels[:, 1:]
    mask = labels != -100
    correct = (preds[mask] == labels[mask]).sum()
    total = mask.sum()
    return {"token_accuracy": float(correct / total) if total > 0 else 0.0}

In [ ]:
# Instantiate the trainer
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=eval_dataset_tokenized,
    data_collator=causal_collator,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=CONFIG["training"]["early_stopping_patience"],
        ),
    ],
)

In [ ]:
# Start the training
trainer.train()

In [ ]:
# Load training log
log_history = trainer.state.log_history
df_log = pd.DataFrame(log_history)

In [ ]:
# Quick visual inspection of the loss on the train dataset
df_log[df_log["loss"].notna()].plot(x="step", y="loss");

In [ ]:
# Compute final perplexity if eval_loss exists
if "eval_loss" in df_log.columns:
    final_eval_loss = df_log["eval_loss"].dropna().iloc[-1]
    print(
        f"Final eval loss: {final_eval_loss:.4f}, "
        f"Perplexity: {np.exp(final_eval_loss):.2f}",
    )

In [ ]:
# Save the finetuned adapter, tokenizer, and training metadata
output_dir = Path(CONFIG["paths"]["ft_model"])
output_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# Persist training configuration so the evaluation notebook knows
# which base model and hyperparameters were used
metadata = {
    "base_model": BASE_MODEL,
    "model_config": CONFIG["model"],
    "lora_config": CONFIG["lora"],
    "training_config": CONFIG["training"],
    "seed": SEED,
}
with Path.open(output_dir / "training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Adapter and tokenizer saved to {output_dir}")

## Quick evaluation of the finetuned model

In [ ]:
def generate_response(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    temperature: float = 0.0,
    top_p: float = 1.0,
    **kwargs: dict[str, Any],
) -> str:
    """Return model response for the given prompt.

    Args:
        model: The finetuned language model.
        tokenizer: The tokenizer associated with the model.
        prompt: The input prompt string for generation.
        temperature: The sampling temperature.
        top_p: The nucleus sampling parameter.
        **kwargs: Additional generation parameters passed to ``model.generate()``.

    Returns:
        The generated response string.

    Raises:
        ValueError: If temperature is negative.

    """
    inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(
        model.device,
    )
    inputs_len = inputs["input_ids"].size(1)

    if temperature > 0.0:
        model_params = {
            "do_sample": True,
            "temperature": min(temperature, 1.9999999),
            "top_p": top_p,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
        }
    elif temperature == 0.0:
        model_params = {
            "do_sample": False,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
        }
    else:
        msg = "Temperature must be non-negative."
        raise ValueError(msg)

    # Always request structured output for reliable .sequences access
    kwargs.pop("return_dict_in_generate", None)
    outputs = model.generate(
        **inputs,
        **model_params,
        return_dict_in_generate=True,
        **kwargs,
    )

    return tokenizer.decode(
        outputs.sequences[0][inputs_len:],
        skip_special_tokens=True,
    ).strip()

In [ ]:
# Quick generation test
for i in range(1):
    ex = eval_dataset_formatted[i]
    isys, iprompt = ex["messages"][:-1]
    itext = ex["text"]
    igt = ex["ground_truth"]

    # Remove assistant part for the input (input contains special tokens)
    input_text = itext.split(sep, maxsplit=1)[0] + sep

    # Generate response
    response = generate_response(
        model,
        tokenizer,
        input_text,
        temperature=0.0,
        max_new_tokens=256,
    )

    # Print results
    print(f"{isys['role']}:\n{isys['content']}\n")
    print(f"{iprompt['role']}:\n{iprompt['content']}\n")
    print(f"assistant:\n{response}\n")
    ground_truth_text = "\n".join(igt)
    print(f"ground truth:\n{ground_truth_text}")